# Acoustic-only matched training

Standalone training notebook. All project functions are embedded at build time.


## 1. Run configuration
Set the prepared experiment input, writable output directory, condition, seed and device explicitly.

In [ ]:
from pathlib import Path


EXPERIMENT_PATH = "/explicit/path/to/prepared-experiment"
OUTPUT_ROOT = "/explicit/writable/path/to/runs"
EXPERIMENT_ID = "REPLACE_WITH_EXPLICIT_ID"
DEVICE = "cuda"  # or "cpu"
CONDITION = "mse_only"
SEED = 42
LAMBDA_MSE = 1.0


## 2. Embedded dependencies and data loader
This section contains the model-independent experiment loader, manifest checks, mappings, splits, acoustic targets and matched initialization.

In [ ]:
# GENERATED from src/phonetics; rebuild with meta/tools/build_training_notebooks.py.
# Do not edit this cell by hand.
BUILD_METADATA = {
  "baseline_git_sha": "63dbf68c041c7581ffbe5c389569a805b8408ca2",
  "digest_scope": "UTF-8 selected canonical definitions in dependency order: model, reproducibility, data loader, metrics, training",
  "embedded_definitions_sha256": "84dbbb497f4696169c77c322fd1df8923af040fa703c31cfbf06d2003fd8a89a",
  "source_sha256": {
    "data.py": "f8f6c6c588f11b36de854a7b144883231a4ee29851ce0bcf68bcf3af610933d6",
    "metrics.py": "75da4114d75052791bd24b427a4f8c2952244251c41a7c71cc51fd35cf30e134",
    "model.py": "9f6bd2de95cec891c875280e518726ec1c9f6f0ec6080b55fe564a88170415af",
    "reproducibility.py": "ad47bd3899c536a7d4a8d151469b1d206b15a653bd00e6498eeddb90a37446f3",
    "training.py": "58942905394be2ceb9c981767b9ac13346a3a2a2c65c83ed694249a88d2e83a6"
  }
}
import hashlib
import json
import pickle
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

class ClassifierHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.dropout(self.act(self.fc1(x))))

class Encoder(nn.Module):
    """Byte-for-byte state-dict-compatible extraction of the pure encoder."""

    def __init__(self, vocab_size_tokens: int, embedding_dim: int, hidden_dim: int,
                 num_initials: int, num_finals: int, num_tones: int,
                 num_layers: int = 4, dropout: float = 0.3, mlp_hidden_dim: int = 256):
        super().__init__()
        self.embedding_bag = nn.EmbeddingBag(vocab_size_tokens, embedding_dim, mode="mean")
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, bidirectional=True,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.initial_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_initials, dropout)
        self.final_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_finals, dropout)
        self.tone_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_tones, dropout)

    def forward(self, token_ids, token_offsets, char_lengths, return_hidden=False):
        char_emb = self.embedding_bag(token_ids, token_offsets)
        padded = nn.utils.rnn.pad_sequence(torch.split(char_emb, char_lengths.tolist()), batch_first=True)
        packed = pack_padded_sequence(padded, char_lengths.cpu(), batch_first=True, enforce_sorted=False)
        char_out, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True)
        char_out = self.dropout(char_out)
        result = (self.initial_head(char_out), self.final_head(char_out), self.tone_head(char_out))
        return (*result, char_out) if return_hidden else result

def build_encoder(config: dict, vocab: dict) -> Encoder:
    arch = config["architecture"]
    acoustic_dim = int(config["acoustic_dim"])
    if 2 * int(arch["hidden_dim"]) != acoustic_dim:
        raise ValueError("2 * hidden_dim must equal acoustic_dim for normalized-MSE alignment")
    labels = vocab["labels"]
    return Encoder(len(vocab["wubi_tokens"]), int(arch["embedding_dim"]), int(arch["hidden_dim"]),
                   len(labels["initials"]), len(labels["finals"]), len(labels["tones"]),
                   int(arch["num_layers"]), float(arch["dropout"]), int(arch["mlp_hidden_dim"]))

def seed_everything(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.benchmark = False

def make_generator(seed: int, stream: int = 0) -> torch.Generator:
    """Return an independent generator; module initialization cannot advance it."""
    generator = torch.Generator()
    generator.manual_seed(int(seed) + 1_000_003 * int(stream))
    return generator

PREPARATION_VERSION = "2"

INITIALS = ["", "b", "p", "m", "f", "d", "t", "n", "l", "g", "k", "h", "j", "q", "x", "zh", "ch", "sh", "r", "z", "c", "s"]

FINALS = ["a", "ai", "an", "ang", "ao", "e", "ei", "en", "eng", "er", "i", "ia", "ian", "iang", "iao", "ie", "in", "ing", "iong", "iu", "o", "ong", "ou", "u", "ua", "uai", "uan", "uang", "ui", "un", "uo", "v", "ve", "van", "vn"]

TONES = ["1", "2", "3", "4", "5"]

ZERO_INITIAL = {"yi":"i","ya":"ia","yao":"iao","ye":"ie","you":"iu","yan":"ian","yin":"in","yang":"iang","ying":"ing","yong":"iong","yu":"v","yue":"ve","yuan":"van","yun":"vn","wu":"u","wa":"ua","wo":"uo","wai":"uai","wei":"ui","wan":"uan","wen":"un","wang":"uang","weng":"ong"}

def _sha(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def _decompose(token: str) -> tuple[int, int, int]:
    tone = token[-1] if token[-1:].isdigit() else "5"
    base = token[:-1] if token[-1:].isdigit() else token
    base = ZERO_INITIAL.get(base, base)
    initial = next((x for x in sorted(INITIALS[1:], key=len, reverse=True) if base.startswith(x)), "")
    final = base[len(initial):]
    if initial in {"j", "q", "x"}:
        final = {"u":"v", "ue":"ve", "uan":"van", "un":"vn"}.get(final, final)
    if final in FINALS and tone in TONES:
        return INITIALS.index(initial), FINALS.index(final), TONES.index(tone)
    raise ValueError(f"invalid pinyin decomposition: {token}")

def _encode(text: str, mapping: dict, stoi: dict[str, int]) -> tuple[list[int], list[list[int]]]:
    ids, spans = [], []
    for char in text:
        code = str(mapping.get(char, char))
        start = len(ids)
        ids.extend(stoi.get(symbol, stoi["[UNK]"]) for symbol in code)
        ids.append(stoi["#"])
        spans.append([start, len(ids)])
    return ids, spans

class ExperimentDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, index): return self.samples[index]

def _collate(samples):
    token_ids, offsets, cursor = [], [], 0
    max_chars = max(len(s["character_spans"]) for s in samples)
    component = torch.full((len(samples), max_chars, 3), -100, dtype=torch.long)
    acoustic = torch.full((len(samples), max_chars), -1, dtype=torch.long)
    for row, sample in enumerate(samples):
        token_ids.extend(sample["token_ids"])
        for start, _ in sample["character_spans"]: offsets.append(cursor + start)
        cursor += len(sample["token_ids"])
        n = len(sample["character_spans"]); component[row, :n] = torch.tensor(sample["component_labels"]); acoustic[row, :n] = torch.tensor(sample["acoustic_ids"])
    return {"sample_ids": [s["sample_id"] for s in samples], "token_ids": torch.tensor(token_ids), "token_offsets": torch.tensor(offsets),
            "char_lengths": torch.tensor([len(s["character_spans"]) for s in samples]), "component_labels": component, "acoustic_ids": acoustic}

def load_experiment(path: str | Path) -> dict:
    root = Path(path).resolve(); manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 1 or manifest.get("preparation_version") != PREPARATION_VERSION:
        raise ValueError("unsupported manifest schema/preparation version")
    required = {"samples.jsonl", "vocab.json", "wubi_mapping.pkl", "splits.json", "config.json", "acoustic_targets.pt"}
    required |= {f"initialization/seed-{seed}.pt" for seed in manifest.get("counts", {}).get("seeds", [42])}
    if not required.issubset(manifest.get("artifacts", {})):
        raise ValueError("manifest is missing required artifact hashes")
    for relative, digest in manifest["artifacts"].items():
        artifact = (root / relative).resolve()
        if root not in artifact.parents: raise ValueError(f"unsafe artifact path: {relative}")
        if not artifact.is_file() or _sha(artifact) != digest: raise ValueError(f"artifact checksum mismatch: {relative}")
    config = json.loads((root / "config.json").read_text(encoding="utf-8")); vocab = json.loads((root / "vocab.json").read_text(encoding="utf-8")); splits = json.loads((root / "splits.json").read_text(encoding="utf-8"))
    with (root / "wubi_mapping.pkl").open("rb") as handle: wubi = pickle.load(handle)
    if config.get("schema_version") != 1 or config.get("seeds") != [42] or config.get("split_seed") != 42 or config.get("split_ratios") != [0.8, 0.1, 0.1] or config.get("training", {}).get("epochs") != 20:
        raise ValueError("config violates fixed experiment protocol")
    if set(splits) != {"train", "validation", "test"} or any(not x for x in splits.values()): raise ValueError("all three splits must be non-empty")
    samples = [json.loads(line) for line in (root / "samples.jsonl").read_text(encoding="utf-8").splitlines()]
    by_id = {s["sample_id"]: s for s in samples}; members = [x for split in splits.values() for x in split]
    if len(by_id) != len(samples) or len(members) != len(set(members)) or set(members) != set(by_id): raise ValueError("split coverage/disjointness or sample IDs invalid")
    if len({s["text"] for s in samples}) != len(samples): raise ValueError("duplicate exact text in experiment bundle")
    acoustic = torch.load(root / "acoustic_targets.pt", map_location="cpu", weights_only=True)
    norms = torch.linalg.vector_norm(acoustic.double(), dim=1)
    if acoustic.ndim != 2 or acoustic.shape[0] != len(vocab["acoustic_tokens"]) or acoustic.shape[1] != config["acoustic_dim"] or not torch.isfinite(acoustic).all() or not torch.isfinite(norms).all() or torch.any(norms <= 1e-12): raise ValueError("invalid acoustic target matrix")
    for sample in samples:
        n=len(sample["text"])
        if not sample["text"] or not sample["token_ids"]: raise ValueError("empty text or token IDs")
        if not (n == len(sample["character_spans"]) == len(sample["component_labels"]) == len(sample["acoustic_ids"]) == len(sample["char_ids"])): raise ValueError(f"sample shape mismatch: {sample['sample_id']}")
        if any(i < 0 or i >= acoustic.shape[0] for i in sample["acoustic_ids"]): raise ValueError("acoustic ID out of range")
        expected = hashlib.sha256(json.dumps([sample["text"], [vocab["acoustic_tokens"][i] for i in sample["acoustic_ids"]]], ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()
        if sample["sample_id"] != expected: raise ValueError("sample ID hash mismatch")
        if any(i < 0 or i >= len(vocab["wubi_tokens"]) for i in sample["token_ids"]): raise ValueError("token ID out of range")
        if any(i < 0 or i >= len(vocab["char_tokens"]) for i in sample["char_ids"]): raise ValueError("char ID out of range")
        if any(any(v < 0 or v >= len(vocab["labels"][["initials","finals","tones"][j]]) for j,v in enumerate(row)) for row in sample["component_labels"]): raise ValueError("component ID out of range")
        spans=sample["character_spans"]
        if spans[0][0] != 0 or spans[-1][1] != len(sample["token_ids"]) or any(a >= b or (i and a != spans[i-1][1]) for i,(a,b) in enumerate(spans)): raise ValueError("invalid character spans")
        encoded, encoded_spans = _encode(sample["text"], wubi, {v:i for i,v in enumerate(vocab["wubi_tokens"])})
        if encoded != sample["token_ids"] or encoded_spans != spans: raise ValueError("text encoding disagrees with bundled Wubi mapping")
        expected_components = [list(_decompose(vocab["acoustic_tokens"][i])) for i in sample["acoustic_ids"]]
        if sample["component_labels"] != expected_components: raise ValueError("component labels disagree with acoustic labels")
    train_chars = sorted({ch for sid in splits["train"] for ch in by_id[sid]["text"]})
    if vocab["char_tokens"] != ["[PAD]", "[UNK]", *train_chars]: raise ValueError("character vocabulary is not exactly train-derived")
    char_stoi = {char:i for i,char in enumerate(vocab["char_tokens"])}
    for sample in samples:
        if sample["char_ids"] != [char_stoi.get(ch, 1) for ch in sample["text"]]: raise ValueError("character IDs disagree with train-derived vocabulary")
    checkpoints = {}
    for seed in config["seeds"]:
        checkpoint = torch.load(root / f"initialization/seed-{seed}.pt", map_location="cpu", weights_only=True)
        expected_sources = {k:v["sha256"] for k,v in manifest["sources"].items()}
        if checkpoint["seed"] != seed or checkpoint["config_sha256"] != _sha(root / "config.json") or checkpoint["vocab_sha256"] != _sha(root / "vocab.json") or checkpoint.get("source_sha256") != expected_sources: raise ValueError("initialization provenance mismatch")
        model = build_encoder(config, vocab); model.load_state_dict(checkpoint["state_dict"], strict=True); checkpoints[int(seed)] = checkpoint
    return {"root": root, "manifest": manifest, "config": config, "vocab": vocab, "splits": splits, "samples": by_id, "acoustic_targets": acoustic, "initializations": checkpoints}

def create_loaders(experiment: dict, seed: int, batch_size: int | None = None, num_workers: int = 0) -> dict[str, DataLoader]:
    batch = batch_size or int(experiment["config"]["training"]["batch_size"])
    result = {}
    for stream, split in enumerate(("train", "validation", "test"), start=1):
        dataset = ExperimentDataset([experiment["samples"][x] for x in experiment["splits"][split]])
        result[split] = DataLoader(dataset, batch_size=batch, shuffle=split == "train", generator=make_generator(seed, stream), num_workers=num_workers, collate_fn=_collate)
    return result

EPS = 1e-12

def normalize_vectors(vectors: torch.Tensor) -> torch.Tensor:
    if not torch.isfinite(vectors).all(): raise ValueError("vectors contain non-finite values")
    norms = torch.linalg.vector_norm(vectors.double(), dim=-1)
    if not torch.isfinite(norms).all() or torch.any(norms <= EPS): raise ValueError("vectors have zero, tiny, or overflowed norms")
    return vectors.double() / norms.unsqueeze(-1)

def batch_statistics(outputs, batch, acoustic_targets, condition: str, lambda_mse: float = 1.0):
    initial, final, tone, hidden = outputs
    labels = batch["component_labels"]
    mask = labels[..., 0] != -100
    count = int(mask.sum())
    if count == 0: raise ValueError("batch contains no valid characters")
    if not torch.equal(mask, batch["acoustic_ids"] >= 0): raise ValueError("classification and acoustic masks differ")
    if condition == "combined" and (not torch.isfinite(torch.tensor(lambda_mse)) or lambda_mse < 0): raise ValueError("lambda_mse must be finite and nonnegative")
    ce_sum = None if condition == "mse_only" else sum(F.cross_entropy(logits[mask], labels[..., i][mask], reduction="sum") for i, logits in enumerate((initial, final, tone)))
    predicted = normalize_vectors(hidden[mask]); target = normalize_vectors(acoustic_targets[batch["acoustic_ids"][mask]])
    squared = (predicted - target).square().mean(dim=-1)
    delta = predicted - target; distance = torch.linalg.vector_norm(delta, dim=-1)
    mse_sum = squared.sum(); distance_sum = distance.sum(); squared_distance_sum = distance.square().sum()
    l1_sum = delta.abs().sum(dim=-1).sum(); rmse_sum = torch.sqrt(squared).sum()
    cosine_direct_sum = (predicted * target).sum(dim=-1).sum()
    cosine_from_squared_sum = (1.0 - 0.5 * distance.square()).sum()
    if condition == "ce_only": objective_sum = ce_sum
    elif condition == "mse_only": objective_sum = mse_sum
    elif condition == "combined": objective_sum = ce_sum + float(lambda_mse) * mse_sum
    else: raise ValueError(f"unknown condition: {condition}")
    stats = {"characters": count, "objective_sum": objective_sum, "mse_sum": mse_sum, "distance_sum": distance_sum,
             "squared_distance_sum": squared_distance_sum, "l1_sum": l1_sum, "rmse_sum": rmse_sum,
             "cosine_direct_sum": cosine_direct_sum, "cosine_from_squared_sum": cosine_from_squared_sum}
    if condition != "mse_only":
        stats["ce_sum"] = ce_sum
        stats.update({f"{name}_correct": (logits[mask].argmax(-1) == labels[..., i][mask]).sum()
                      for i, (name, logits) in enumerate(zip(("initial", "final", "tone"), (initial, final, tone)))})
        predictions = torch.stack([x[mask].argmax(-1) for x in (initial, final, tone)], dim=-1)
        stats["exact_correct"] = (predictions == labels[mask]).all(dim=-1).sum()
    return stats

def finalize_statistics(totals: dict, condition: str) -> dict:
    count = totals.get("characters", 0)
    if count <= 0: raise ValueError("loader contains no valid characters")
    result = {"characters": count, "num_characters": count, "objective": totals["objective_sum"] / count,
              "normalized_mse": totals["mse_sum"] / count, "avg_mse": totals["mse_sum"] / count,
              "avg_squared_distance": totals["squared_distance_sum"] / count,
              "avg_distance": totals["distance_sum"] / count, "avg_l1_distance": totals["l1_sum"] / count,
              "avg_rmse": totals["rmse_sum"] / count, "avg_cosine_direct": totals["cosine_direct_sum"] / count,
              "avg_cosine_from_squared_distance": totals["cosine_from_squared_sum"] / count}
    if condition != "mse_only":
        result["ce"] = totals["ce_sum"] / count
        result["exact_accuracy"] = totals["exact_correct"] / count
        for name in ("initial", "final", "tone"): result[f"{name}_accuracy"] = totals[f"{name}_correct"] / count
    return result

def load_initialized_model(experiment: dict, seed: int, device="cpu"):
    if seed not in experiment["initializations"]: raise ValueError("seed has no checked initialization")
    model = build_encoder(experiment["config"], experiment["vocab"])
    model.load_state_dict(experiment["initializations"][seed]["state_dict"], strict=True)
    return model.to(device)

def _move(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k,v in batch.items()}

def _epoch(model, loader, acoustic, condition, lambda_mse, device, optimizer=None):
    model.train(optimizer is not None); totals = {}
    for batch in loader:
        batch = _move(batch, device)
        if optimizer: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(optimizer is not None):
            outputs = model(batch["token_ids"], batch["token_offsets"], batch["char_lengths"], return_hidden=True)
            stats = batch_statistics(outputs, batch, acoustic, condition, lambda_mse)
            loss = stats["objective_sum"] / stats["characters"]
            if optimizer: loss.backward(); optimizer.step()
        for key, value in stats.items(): totals[key] = totals.get(key, 0.0) + (value.detach().item() if torch.is_tensor(value) else value)
    return finalize_statistics(totals, condition)

def train_experiment(experiment_path, *, condition: str, seed: int = 42, lambda_mse: float = 1.0,
                     runs_root=None, device=None, code_provenance: dict | None = None):
    if condition not in {"ce_only", "mse_only", "combined"}: raise ValueError("invalid condition")
    if condition != "combined" and lambda_mse != 1.0: raise ValueError("lambda_mse only applies to combined")
    if condition == "combined" and (not torch.isfinite(torch.tensor(lambda_mse)) or lambda_mse < 0): raise ValueError("lambda_mse must be finite and nonnegative")
    experiment = load_experiment(experiment_path); config = experiment["config"]; device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model = load_initialized_model(experiment, seed, device)
    seed_everything(seed)
    loaders = create_loaders(experiment, seed, num_workers=0)
    if condition == "mse_only":
        for head in (model.initial_head, model.final_head, model.tone_head):
            for parameter in head.parameters(): parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=config["training"]["learning_rate"], weight_decay=config["training"]["weight_decay"])
    suffix = Path(condition) / (f"lambda-{lambda_mse:g}" if condition == "combined" else "") / f"seed-{seed}"
    runs_base = Path(runs_root).resolve() if runs_root else experiment["root"].parent.parent / "runs"
    base = (runs_base / experiment["manifest"]["experiment_id"]).resolve()
    run = (base / suffix).resolve()
    if base not in run.parents or run.exists(): raise FileExistsError(f"unsafe or existing run directory: {run}")
    run.mkdir(parents=True)
    acoustic = experiment["acoustic_targets"].to(device); history=[]; best=float("inf")
    try:
        for epoch in range(config["training"]["epochs"]):
            train_metrics = _epoch(model, loaders["train"], acoustic, condition, lambda_mse, device, optimizer)
            validation_metrics = _epoch(model, loaders["validation"], acoustic, condition, lambda_mse, device)
            history.append({"epoch": epoch + 1, "train": train_metrics, "validation": validation_metrics})
            if validation_metrics["objective"] < best:
                best = validation_metrics["objective"]
                torch.save({"state_dict": model.state_dict(), "seed": seed, "condition": condition, "lambda_mse": lambda_mse,
                            "manifest_sha256": _sha(experiment["root"] / "manifest.json"), "config_sha256": _sha(experiment["root"] / "config.json"),
                            "vocab_sha256": _sha(experiment["root"] / "vocab.json")}, run / "best.pt")
        best_checkpoint = torch.load(run / "best.pt", map_location=device, weights_only=True); model.load_state_dict(best_checkpoint["state_dict"], strict=True)
        test_metrics = _epoch(model, loaders["test"], acoustic, condition, lambda_mse, device)
        actual = {"experiment_id": experiment["manifest"]["experiment_id"], "seed": seed, "condition": condition, "lambda_mse": lambda_mse,
                  "device": str(device), "manifest_sha256": best_checkpoint["manifest_sha256"], "config": config,
                  "code_sha256": code_provenance if code_provenance is not None else {
                      p.name: _sha(p) for p in (Path(__file__), Path(__file__).with_name("metrics.py"),
                          Path(__file__).with_name("data.py"), Path(__file__).with_name("model.py"),
                          Path(__file__).with_name("reproducibility.py"))}}
        (run / "history.json").write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")
        (run / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2) + "\n", encoding="utf-8")
        (run / "actual_config.json").write_text(json.dumps(actual, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    except Exception:
        raise
    return {"run_dir": run, "history": history, "test_metrics": test_metrics}



## 3. Load the prepared experiment
The bundle is generated before this notebook runs; training never rebuilds splits or vocabulary.

In [ ]:
experiment_path = Path(EXPERIMENT_PATH).expanduser().resolve()
output_root = Path(OUTPUT_ROOT).expanduser().resolve()
if experiment_path.name != EXPERIMENT_ID:
    raise ValueError("EXPERIMENT_PATH basename must equal EXPERIMENT_ID")
experiment = load_experiment(experiment_path)
loaders = create_loaders(experiment, SEED, num_workers=0)
print({
    "experiment_id": experiment["manifest"]["experiment_id"],
    "splits": {name: len(loader.dataset) for name, loader in loaders.items()},
    "input": str(experiment_path),
    "output": str(output_root),
})


## 4. Model, objectives and training loop
The embedded definitions implement the canonical model, RNG streams, objectives, validation and checkpoint selection.

In [ ]:
result = train_experiment(
    experiment_path,
    condition=CONDITION,
    seed=SEED,
    lambda_mse=LAMBDA_MSE,
    runs_root=output_root,
    device=DEVICE,
    code_provenance=BUILD_METADATA,
)
result["test_metrics"]


## 5. Test results
The training cell returns test metrics from the selected checkpoint.

In [ ]:
test_metrics = result["test_metrics"]
test_metrics